# Rates — Simple CRUD Test

Insert, read, update, and delete rows from `[rates].[dim_curve]` and `[rates].[fact_observation]`.

In [1]:
from datetime import datetime, timezone

import pandas as pd
from sqlalchemy import text

from imdr.config.settings import get_settings
from imdr.connectors.mssql import MSSQLConnector

connector = MSSQLConnector(get_settings())
print("Connected:", connector.engine.url)

Connected: mssql+pyodbc://@rv-database-1.ctym72ljvrjq.ap-southeast-1.rds.amazonaws.com:1433/IMDR?Trusted_Connection=yes&driver=SQL+Server


## 1. READ — Check existing dim_curve rows

In [5]:
with connector.session() as session:
    df_curves = pd.read_sql(
        text("SELECT * FROM rates.fact_observation"),
        session.connection(),
    )

print(f"rates.fact_observation rows: {len(df_curves)}")
df_curves.head(10)

rates.fact_observation rows: 1308


,id,curve_id,ts,quote,tenor,value,created_at,updated_at
0,1,1,2026-03-09 00:00:00.0000000 +00:00,par,10M,3.55676,2026-03-11 02:32:47.9127973 +08:00,2026-03-11 07:02:10.7532383 +08:00
1,2,1,2026-03-09 00:00:00.0000000 +00:00,par,10Y,3.68369,2026-03-11 02:32:47.9127973 +08:00,2026-03-11 07:02:10.7532383 +08:00
2,3,1,2026-03-09 00:00:00.0000000 +00:00,par,11M,3.53989,2026-03-11 02:32:47.9127973 +08:00,2026-03-11 07:02:10.7532383 +08:00
3,4,1,2026-03-09 00:00:00.0000000 +00:00,par,11Y,3.73177,2026-03-11 02:32:47.9127973 +08:00,2026-03-11 07:02:10.7532383 +08:00
4,5,1,2026-03-09 00:00:00.0000000 +00:00,par,12Y,3.77827,2026-03-11 02:32:47.9127973 +08:00,2026-03-11 07:02:10.7532383 +08:00
5,6,1,2026-03-09 00:00:00.0000000 +00:00,par,13Y,3.82147,2026-03-11 02:32:47.9127973 +08:00,2026-03-11 07:02:10.7532383 +08:00
6,7,1,2026-03-09 00:00:00.0000000 +00:00,par,14Y,3.85996,2026-03-11 02:32:47.9127973 +08:00,2026-03-11 07:02:10.7532383 +08:00
7,8,1,2026-03-09 00:00:00.0000000 +00:00,par,15M,3.46718,2026-03-11 02:32:47.9127973 +08:00,2026-03-11 07:02:10.7532383 +08:00
8,9,1,2026-03-09 00:00:00.0000000 +00:00,par,15Y,3.89223,2026-03-11 02:32:47.9127973 +08:00,2026-03-11 07:02:10.7532383 +08:00
9,10,1,2026-03-09 00:00:00.0000000 +00:00,par,16Y,3.91838,2026-03-11 02:32:47.9127973 +08:00,2026-03-11 07:02:10.7532383 +08:00


## 2. CREATE — Insert a test observation

Pick the first active curve and insert a dummy par observation.

In [ ]:
# Use the first active curve_id from dim_curve
test_curve_id = int(df_curves.loc[df_curves["curve_status"] == "active", "id"].iloc[0])
test_ccy = df_curves.loc[df_curves["id"] == test_curve_id, "ccy"].iloc[0]
test_curve = df_curves.loc[df_curves["id"] == test_curve_id, "curve"].iloc[0]
print(f"Using curve_id={test_curve_id} ({test_ccy} {test_curve})")

insert_sql = text("""
    INSERT INTO rates.fact_observation (curve_id, ts, quote, tenor, value)
    OUTPUT INSERTED.id
    VALUES (:curve_id, :ts, :quote, :tenor, :value)
""")

params = dict(
    curve_id=test_curve_id,
    ts="2099-01-01T00:00:00+00:00",
    quote="par",
    tenor="5Y",
    value=3.85,
)

with connector.session() as session:
    result = session.execute(insert_sql, params)
    inserted_id = result.scalar()

print(f"Inserted row with id = {inserted_id}")

## 3. READ — Fetch the test row back

In [ ]:
read_sql = text("""
    SELECT o.id, c.ccy, c.curve, o.ts, o.quote, o.tenor, o.value, o.created_at
    FROM rates.fact_observation o
    JOIN rates.dim_curve c ON o.curve_id = c.id
    WHERE o.id = :id
""")

with connector.session() as session:
    df_row = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

df_row

## 4. UPDATE — Change the value

In [ ]:
update_sql = text("""
    UPDATE rates.fact_observation
    SET value = :value
    WHERE id = :id
""")

with connector.session() as session:
    session.execute(update_sql, {"value": 4.25, "id": inserted_id})

# Verify
with connector.session() as session:
    df_updated = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

print(f"Updated value = {df_updated['value'].iloc[0]}")
df_updated

## 5. DELETE — Remove the test row

In [ ]:
delete_sql = text("DELETE FROM rates.fact_observation WHERE id = :id")

with connector.session() as session:
    result = session.execute(delete_sql, {"id": inserted_id})

print(f"Deleted {result.rowcount} row(s)")

# Verify
with connector.session() as session:
    df_check = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

print(f"Rows remaining for id={inserted_id}: {len(df_check)}")

In [ ]:
connector.dispose()
print("Done — connection pool closed.")